# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

**Recommended:** create a venv, then `pip install -r requirements.txt` from the repo root, and register/select a Jupyter kernel for that venv (see `context/ENVIRONMENT_SETUP.md`).

Optional: the cell below uses `uv` and shell paths that target **Linux/macOS** (and `wget`). On **Windows**, prefer `requirements.txt` + the correct kernel instead of running the install cell.

> **After first install, restart the kernel** so packages (especially `torch`, `vllm` if installed) are picked up.

### Optional install cell (Linux/macOS)

Uncomment commands in the next cell if you use `uv`; otherwise rely on `pip install -r requirements.txt`.

In [1]:
# Install deps into the *same* Python as this notebook (avoids wrong paths on Windows / wrong cwd).
# Optional `uv` lines at bottom: Linux/macOS only. On Windows use sys.executable or .venv\Scripts\python.exe.
from pathlib import Path
import sys


def _repo_root() -> Path:
    p = Path.cwd().resolve()
    for d in (p, *p.parents):
        if (d / "judger.py").is_file():
            return d
    return p


ROOT = _repo_root()
REQ = ROOT / "requirements.txt"

print("Repo root:", ROOT)
print("requirements.txt:", REQ, "| exists:", REQ.is_file())
print("This kernel's Python:", sys.executable)
print()

if REQ.is_file():
    print("Install (copy into a terminal, or set RUN_PIP_INSTALL_HERE = True below):")
    print(f'  "{sys.executable}" -m pip install -U pip')
    print(f'  "{sys.executable}" -m pip install -r "{REQ}"')
    print()
    print("Register kernel (after venv is active / using the Python you want):")
    print(f'  "{sys.executable}" -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"')
else:
    print("Could not find requirements.txt next to judger.py — check repo layout.")

# One-shot install into *this* Jupyter kernel (same as running the pip lines above)
RUN_PIP_INSTALL_HERE = False
if RUN_PIP_INSTALL_HERE and REQ.is_file():
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pip"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(REQ)])
    print("Done. Restart the kernel, then continue.")

# Linux/macOS only (optional):
# !wget -qO- https://astral.sh/uv/install.sh | sh
# !uv venv .venv --seed
# !uv pip install -r requirements.txt

Repo root: /mnt/c/Users/sardo/OneDrive/Desktop/Classes/projects/math-qa-llm
requirements.txt: /mnt/c/Users/sardo/OneDrive/Desktop/Classes/projects/math-qa-llm/requirements.txt | exists: True
This kernel's Python: /home/sardo/cse151b-venv/bin/python

Install (copy into a terminal, or set RUN_PIP_INSTALL_HERE = True below):
  "/home/sardo/cse151b-venv/bin/python" -m pip install -U pip
  "/home/sardo/cse151b-venv/bin/python" -m pip install -r "/mnt/c/Users/sardo/OneDrive/Desktop/Classes/projects/math-qa-llm/requirements.txt"

Register kernel (after venv is active / using the Python you want):
  "/home/sardo/cse151b-venv/bin/python" -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"


### Kernel note

Shell activation (`source .venv/...`) does **not** change which Python Jupyter uses. Pick the project venv **kernel** instead.

In [2]:
# No-op: Jupyter runs the interpreter of the *selected kernel*, not the shell you activate here.
print("Using:", __import__("sys").executable)

Using: /home/sardo/cse151b-venv/bin/python


In [ ]:
# ── Google Colab Setup ──────────────────────────────────────────────────────
# This cell is a no-op when running locally.  On Colab it installs the
# required packages and mounts Google Drive so checkpoints survive across
# sessions.  Run it first, then restart the runtime if prompted.
import sys, importlib

try:
    import google.colab          # noqa: F401
    _IS_COLAB = True
except ImportError:
    _IS_COLAB = False

if _IS_COLAB:
    print("Colab detected — installing packages (takes ~3-5 min)...")
    import subprocess
    pkgs = [
        "vllm>=0.6.0",
        "transformers>=4.45.0",
        "bitsandbytes>=0.43.0",
        "peft>=0.12.0",
        "trl>=0.12.0",
        "datasets",
        "accelerate>=0.34.0",
        "sympy>=1.12",
        "tqdm",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + pkgs)
    print("Packages installed. Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    print("Drive mounted at /content/drive")
    print()
    print("Next: upload public.jsonl and private.jsonl to")
    print("  MyDrive/math-qa-llm/data/raw/  on your Google Drive.")
    print("Then continue running cells below.")
else:
    print("Running locally — skipping Colab setup.")


## 2. Imports & Configuration

Paths are resolved from the repo root (folder containing `judger.py`), so the notebook works whether your **current working directory** is the repo root or `notebooks/`.

- `DATA_PATH` — default `data/raw/public.jsonl` (labeled)
- `PRIVATE_PATH` — `data/raw/private.jsonl` (switch `DATA_PATH` to this for submission-only runs; no local accuracy)
- `OUTPUT_PATH` — default under `artifacts/logs/runs/`
- `N_QUESTIONS` — `None` = full file; integer = smoke-test subset (must match generation + scoring)
- `TEST_RANDOM_SUBSET` — if `True` and `N_QUESTIONS` is an int, draw a **random** sample (see `RANDOM_SEED`) instead of the **first** N rows — better for quick sanity checks across the file
- `GPU_ID` — passed to `CUDA_VISIBLE_DEVICES` (single GPU is usually `"0"`)
- `MAX_TOKENS` — max **new** tokens per answer; Qwen3-Thinking needs headroom for long `<think>` chains before `\boxed{}` (default **8192**; **16384** safer on hard items, slower)

In [ ]:
import json
import os
import random
import re
import sys
from pathlib import Path
from typing import Optional


def repo_root() -> Path:
    p = Path.cwd().resolve()
    for d in (p, *p.parents):
        if (d / "judger.py").is_file():
            return d
    return p


REPO_ROOT = repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID   = "0"

PUBLIC_PATH  = REPO_ROOT / "data" / "raw" / "public.jsonl"
PRIVATE_PATH = REPO_ROOT / "data" / "raw" / "private.jsonl"

DATA_MODE = "public"
DATA_PATH = PUBLIC_PATH if DATA_MODE == "public" else PRIVATE_PATH

N_QUESTIONS        = None   # None = all questions; set to 50 for a quick smoke test
TEST_RANDOM_SUBSET = False  # only relevant when N_QUESTIONS is set
RANDOM_SEED        = 42

RUN_NAME        = f"adaptive_{DATA_MODE}_v2"
OUTPUT_PATH     = REPO_ROOT / "artifacts" / "logs" / "runs" / f"{RUN_NAME}_results.jsonl"
CHECKPOINT_PATH = REPO_ROOT / "artifacts" / "logs" / "runs" / f"{RUN_NAME}_checkpoint.jsonl"

PHASE1_THINKING_BUDGET    = 4096
PHASE1_MAX_TOKENS         = 6144
PHASE1_N_SAMPLES          = 1

PHASE2_THINKING_BUDGET    = 4096
PHASE2_MAX_TOKENS         = 6144
PHASE2_N_SAMPLES          = 8
PHASE2_TEMPERATURE        = 0.65
PHASE2_REPETITION_PENALTY = 1.05

MAX_TOKENS = PHASE2_MAX_TOKENS

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

from transformers import AutoTokenizer

try:
    from vllm import LLM, SamplingParams
    VLLM_AVAILABLE = True
except ImportError:
    LLM            = None
    SamplingParams = None
    VLLM_AVAILABLE = False

from tqdm.auto import tqdm
# Detect execution environment
try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

USE_VLLM = True

# On Colab: redirect data + artifacts to Google Drive for persistence
if IS_COLAB:
    DRIVE_BASE      = Path('/content/drive/MyDrive/math-qa-llm')
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    PUBLIC_PATH     = DRIVE_BASE / "data" / "raw" / "public.jsonl"
    PRIVATE_PATH    = DRIVE_BASE / "data" / "raw" / "private.jsonl"
    DATA_PATH       = PUBLIC_PATH if DATA_MODE == "public" else PRIVATE_PATH
    OUTPUT_PATH     = DRIVE_BASE / "artifacts" / "logs" / "runs" / f"{RUN_NAME}_results.jsonl"
    CHECKPOINT_PATH = DRIVE_BASE / "artifacts" / "logs" / "runs" / f"{RUN_NAME}_checkpoint.jsonl"
    (DRIVE_BASE / "artifacts" / "logs" / "runs").mkdir(parents=True, exist_ok=True)

print("IS_COLAB   :", IS_COLAB)
print("REPO_ROOT  :", REPO_ROOT)
print("DATA_MODE  :", DATA_MODE)
print("DATA_PATH  :", DATA_PATH, "| exists:", DATA_PATH.is_file())
print("N_QUESTIONS:", N_QUESTIONS)
print("RUN_NAME   :", RUN_NAME)
print("vLLM       :", VLLM_AVAILABLE)


## 3. Load the Dataset

Data is **JSONL**: one JSON object per line. Paths in this notebook default to **`data/raw/public.jsonl`** (development) and **`data/raw/private.jsonl`** (submission); only the public file includes labels.

### Fields (by split)

| Field | Public (`public.jsonl`) | Private (`private.jsonl`) |
|------|-------------------------|---------------------------|
| `id` | Integer, unique per problem | Same |
| `question` | Problem text, **LaTeX**; free-form items use **`[ANS]`** placeholders where answers go | Same |
| `options` | Present for **MCQ** (list of LaTeX choices); omitted for **free-form** | Same rule |
| `answer` | **Present**: MCQ → single capital letter string (e.g. `"C"`); free-form → **list of strings** (one entry per `[ANS]`, in order) | **Omitted** (withheld for evaluation) |

### Formats

- **Free-form:** One or more answers; every sub-answer must match for the problem to count as correct when scoring locally.
- **MCQ:** Model should pick one letter matching the labeled option.

Use **`DATA_PATH`** in the config cell to point at public vs private; run scoring (§7) only when every loaded row has an **`answer`** field.

In [ ]:
with open(DATA_PATH, encoding="utf-8") as f:
    data = [json.loads(line) for line in f]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = len(data) - n_mcq
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

if N_QUESTIONS is None:
    data_run = data
else:
    k = min(int(N_QUESTIONS), len(data))
    if TEST_RANDOM_SUBSET:
        rng      = random.Random(RANDOM_SEED)
        data_run = rng.sample(data, k=k)
    else:
        data_run = data[:k]

_nrun_mcq = sum(bool(d.get("options")) for d in data_run)
print(f"Running on {len(data_run)} questions  ({_nrun_mcq} MCQ, {len(data_run)-_nrun_mcq} free-form)")


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [ ]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician with deep knowledge of all areas of mathematics, "
    "from algebra and calculus to number theory and combinatorics. "
    "This problem is very important to my career - please think carefully and be precise.\n\n"
    "Solve using this structured approach:\n"
    "1. UNDERSTAND: Identify what is given and what you need to find.\n"
    "2. PLAN: Write down the key equations, formulas, or theorems you will use.\n"
    "3. SOLVE: Work through each step carefully. Compute intermediate results explicitly. "
    "Pay special attention to arithmetic - do not skip steps.\n"
    "4. VERIFY: Check that your answer satisfies all conditions in the problem. "
    "Check units, sign, and order of magnitude.\n"
    "5. ANSWER: Put your final answer in \\boxed{}.\n\n"
    "Additional rules:\n"
    "- If the problem has multiple blanks ([ANS] placeholders), put ALL answers "
    "comma-separated in ONE \\boxed{} in the order they appear. "
    "Example: \\boxed{3, -7, 42}.\n"
    "- Simplify all fractions and radical expressions completely.\n"
    "- You'd better be sure of your answer."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician with deep knowledge of all areas of mathematics. "
    "This problem is very important to my career - please think carefully and be precise.\n\n"
    "Solve using this structured approach:\n"
    "1. UNDERSTAND: Read the problem and all answer choices carefully.\n"
    "2. PLAN: Identify the relevant concepts, formulas, or theorems that apply.\n"
    "3. SOLVE: Work through the problem step by step. Compute intermediate results "
    "explicitly - do not skip arithmetic steps.\n"
    "4. ELIMINATE: Cross out answer choices that are clearly wrong.\n"
    "5. VERIFY: Confirm your chosen answer is consistent with every condition in the problem.\n"
    "6. ANSWER: On the very last line of your response, write ONLY \\boxed{X} "
    "where X is the letter of the correct answer (A-J). "
    "Do not write any text after \\boxed{}.\n\n"
    "You'd better be sure of your answer."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    n_ans = question.count("[ANS]")
    if n_ans > 1:
        hint = (
            f"\n\n[Note: This problem has {n_ans} answers. "
            f"Put all {n_ans} answers comma-separated in ONE \\boxed{{}} "
            f"in the order they appear in the question.]"
        )
        return SYSTEM_PROMPT_MATH, question + hint
    return SYSTEM_PROMPT_MATH, question


## 5. Load Model with vLLM (for general case, vLLM is faster)

Model id comes from **`MODEL_ID`** in the config cell (default **Qwen3-8B-Thinking**). We use **BitsAndBytes INT8** in vLLM (`load_format="bitsandbytes"`), which roughly halves GPU memory versus BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel; keep it >= `N_SAMPLES` for self-consistency

In [ ]:
from collections import Counter
from math import ceil


def extract_boxed(text: str) -> str:
    """Extract last \boxed{...} content (nested braces supported)."""
    matches = []
    needle  = r"\boxed{"
    i = 0
    while i < len(text):
        idx = text.find(needle, i)
        if idx == -1:
            break
        j, depth, start = idx + len(needle), 1, idx + len(needle)
        while j < len(text) and depth:
            if text[j] == "{":
                depth += 1
            elif text[j] == "}":
                depth -= 1
                if depth == 0:
                    matches.append(text[start:j])
                    break
            j += 1
        i = idx + 1
    return matches[-1].strip() if matches else ""


def strip_thinking(text: str) -> str:
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()


def is_uncertain(response: str, finish_reason: str = "") -> bool:
    if "length" in str(finish_reason).lower():
        return True
    if not extract_boxed(response):
        return True
    if len(strip_thinking(response)) < 30:
        return True
    return False


def choose_best_sample(samples: list, finish_reasons: list) -> dict:
    """Majority vote; tie-break by longest trace."""
    extracted = [extract_boxed(s) for s in samples]
    nonempty  = [e for e in extracted if e]
    if nonempty:
        counts       = Counter(nonempty)
        top_count    = counts.most_common(1)[0][1]
        tied_answers = {ans for ans, cnt in counts.items() if cnt == top_count}
        candidates   = [i for i, ans in enumerate(extracted) if ans in tied_answers]
        best_idx     = max(candidates, key=lambda i: len(samples[i]))
        best_answer  = extracted[best_idx]
    else:
        top_count, best_idx, best_answer = 0, 0, ""
    uncertain = (
        is_uncertain(samples[best_idx], finish_reasons[best_idx])
        or top_count < ceil(len(samples) / 2)
    )
    return {
        "response":        samples[best_idx],
        "answer":          best_answer,
        "finish_reason":   finish_reasons[best_idx],
        "consensus_count": top_count,
        "n_samples":       len(samples),
        "uncertain":       uncertain,
    }


def make_sampling_params(max_tokens: int, temperature: float = 0.6,
                          repetition_penalty: float = 1.0):
    return SamplingParams(
        max_tokens=max_tokens, temperature=temperature,
        top_p=0.95, top_k=20, min_p=0.0,
        presence_penalty=0.0, repetition_penalty=repetition_penalty,
    )


def build_chat_prompt(item: dict, thinking_budget=None,
                       prefix: str = "", suffix: str = "") -> str:
    """Render a prompt string for llm.generate(). Falls back to plain-text budget hint."""
    system, user = build_prompt(item["question"], item.get("options"))
    if prefix:
        user = prefix + user
    if suffix:
        user = user + suffix
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": user},
    ]
    kwargs = dict(tokenize=False, add_generation_prompt=True, enable_thinking=True)
    if thinking_budget is not None:
        kwargs["thinking_budget"] = thinking_budget
    try:
        return tokenizer.apply_chat_template(messages, **kwargs)
    except TypeError as exc:
        if thinking_budget is None:
            raise
        hint = (
            f"Use at most about {thinking_budget} thinking tokens. "
            "Be concise but do not skip necessary arithmetic.\n\n"
        )
        messages[1]["content"] = hint + messages[1]["content"]
        kwargs.pop("thinking_budget", None)
        return tokenizer.apply_chat_template(messages, **kwargs)


def load_checkpoint(path) -> dict:
    if not path.exists():
        return {}
    records = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            records[str(rec["id"])] = rec
    print(f"Checkpoint: {len(records)} records from {path.name}")
    return records


def write_checkpoint(path, records: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for rec in sorted(records.values(), key=lambda r: int(r["id"])):
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")


In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Scale vLLM settings based on available GPU
# A100 (40 GB) on Colab: higher utilization, bigger batches, no quantization needed
# RTX 3070 (8 GB) locally: conservative settings with INT8 quantization
if IS_COLAB:
    _gpu_util  = 0.90
    _max_seqs  = 16
    _batch_tok = 32768
    _quant     = None   # A100 has enough VRAM to run BF16 natively (faster than INT8)
    _load_fmt  = "auto"
else:
    _gpu_util  = 0.78   # RTX 3070 8 GB — leaves ~1.7 GB for WSL2 overhead
    _max_seqs  = 4
    _batch_tok = 4096
    _quant     = "bitsandbytes"
    _load_fmt  = "bitsandbytes"

llm = LLM(
    model=MODEL_ID,
    quantization=_quant,
    load_format=_load_fmt,
    gpu_memory_utilization=_gpu_util,
    max_model_len=16384,
    trust_remote_code=True,
    max_num_seqs=_max_seqs,
    max_num_batched_tokens=_batch_tok,
)

sampling_params = make_sampling_params(MAX_TOKENS, temperature=0.6)

print("Model loaded.")

INFO 05-06 11:31:58 [utils.py:233] non-default args: {'trust_remote_code': True, 'load_format': 'bitsandbytes', 'max_model_len': 8192, 'gpu_memory_utilization': 0.78, 'max_num_batched_tokens': 4096, 'max_num_seqs': 4, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'model': 'Qwen/Qwen3-4B-Thinking-2507'}
INFO 05-06 11:31:59 [model.py:555] Resolved architecture: Qwen3ForCausalLM
INFO 05-06 11:31:59 [model.py:1680] Using max model len 8192
INFO 05-06 11:31:59 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=4096.
INFO 05-06 11:32:01 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 05-06 11:32:01 [nixl_utils.py:34] NIXL is not available
WARNING 05-06 11:32:01 [nixl_utils.py:44] NIXL agent config is not available
INFO 05-06 11:32:01 [vllm.py:840] Asynchronous scheduling is enabled.
INFO 05-06 11:32:01 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorit

(EngineCore pid=1221876) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


(EngineCore pid=1221876) INFO 05-06 11:32:27 [weight_utils.py:904] Filesystem type for checkpoints: EXT4. Checkpoint size: 7.49 GiB. Available RAM: 9.12 GiB.
(EngineCore pid=1221876) INFO 05-06 11:32:27 [weight_utils.py:927] Auto-prefetch is disabled because the filesystem (EXT4) is not a recognized network FS (NFS/Lustre). If you want to force prefetching, start vLLM with --safetensors-load-strategy=prefetch.


Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]
(EngineCore pid=1221876) /home/sardo/cse151b-venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
(EngineCore pid=1221876)   torch._check_is_size(blocksize)
Loading safetensors checkpoint shards:  33% Completed | 1/3 [00:07<00:14,  7.50s/it]
Loading safetensors checkpoint shards:  67% Completed | 2/3 [00:21<00:11, 11.32s/it]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:22<00:00,  6.48s/it]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:22<00:00,  7.41s/it]
(EngineCore pid=1221876) 


(EngineCore pid=1221876) INFO 05-06 11:32:50 [gpu_model_runner.py:4879] Model loading took 2.7 GiB memory and 26.208087 seconds
(EngineCore pid=1221876) INFO 05-06 11:33:06 [backends.py:1069] Using cache directory: /home/sardo/.cache/vllm/torch_compile_cache/d93cbf8708/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=1221876) INFO 05-06 11:33:06 [backends.py:1128] Dynamo bytecode transform time: 16.23 s
(EngineCore pid=1221876) INFO 05-06 11:33:17 [backends.py:376] Cache the graph of compile range (1, 4096) for later use
(EngineCore pid=1221876) INFO 05-06 11:33:24 [backends.py:391] Compiling a graph for compile range (1, 4096) takes 19.54 s
(EngineCore pid=1221876) INFO 05-06 11:33:31 [decorators.py:668] saved AOT compiled function to /home/sardo/.cache/vllm/torch_compile_cache/torch_aot_compile/72a5432f81c4ff9496d1b344dc16a4adcd0edd151c4390de2e12beeafa32963f/rank_0_0/model
(EngineCore pid=1221876) INFO 05-06 11:33:31 [monitor.py:53] torch.compile took 43.92 s in total
(Engi

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):  75%|███████▌  | 3/4 [00:00<00:00,  3.92it/s]/home/sardo/cse151b-venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
(EngineCore pid=1221876)   torch._check_is_size(blocksize)
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 4/4 [00:01<00:00,  3.71it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 3/3 [00:00<00:00,  4.67it/s]


(EngineCore pid=1221876) INFO 05-06 11:33:42 [gpu_model_runner.py:6133] Graph capturing finished in 3 secs, took 0.10 GiB
(EngineCore pid=1221876) INFO 05-06 11:33:42 [gpu_worker.py:599] CUDA graph pool memory: 0.1 GiB (actual), 0.14 GiB (estimated), difference: 0.04 GiB (36.5%).
(EngineCore pid=1221876) INFO 05-06 11:33:42 [core.py:299] init engine (profile, create kv cache, warmup model) took 52.27 s (compilation: 43.92 s)
(EngineCore pid=1221876) INFO 05-06 11:33:45 [vllm.py:840] Asynchronous scheduling is enabled.
(EngineCore pid=1221876) INFO 05-06 11:33:45 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])
WARNING 05-06 11:33:45 [interface.py:686] Using 'pin_memory=False' as WSL is detected. This may slow down the performance.
Model loaded.


## 5. Load Model with Transformers (alternative to vLLM for DataHub)

We load **Qwen3-4B-Thinking-2507** with **INT4 quantization** via BitsAndBytes.  

Key parameters:
- `load_in_4bit` — quantization strategy of INT4

In [8]:
# --- Transformers load path (disabled when using vLLM — do not run both; GPU OOM) ---
# import torch
# from transformers import AutoModelForCausalLM, BitsAndBytesConfig
#
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = "left"  # required for correct batched generation in decoder-only LMs
#
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )
#
# llm = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     trust_remote_code=True,
#     quantization_config=bnb_config,
#     device_map="auto",
# )
#
# _device = next(llm.parameters()).device
# print("Model device:", _device)


## 6. Generate Responses - Adaptive Multi-Pass

The adaptive path concentrates compute where it is useful:

1. **Phase 1:** one fast response for every question (`thinking_budget=1024`, `max_tokens=2048`).
2. **Phase 2:** retry uncertain questions with more budget and `N=4` majority vote.
3. **Phase 3:** retry the still-uncertain tail with maximum budget and `N=8` majority vote.

The cell writes `CHECKPOINT_PATH` after each phase/batch so interrupted runs can resume. It also assembles `responses` in `data_run` order, so scoring and saving continue to work.


### Generate with vLLM

In [ ]:
response_records = load_checkpoint(CHECKPOINT_PATH)

# Phase 1 — single batched call for all questions
phase1_params  = make_sampling_params(PHASE1_MAX_TOKENS, temperature=0.6)
missing_phase1 = [item for item in data_run if str(item["id"]) not in response_records]

if missing_phase1:
    print(f"Phase 1: {len(missing_phase1)} questions "
          f"({len(data_run)-len(missing_phase1)} checkpointed)")
    phase1_prompts = [
        build_chat_prompt(item, thinking_budget=PHASE1_THINKING_BUDGET)
        for item in missing_phase1
    ]
    phase1_outputs = llm.generate(phase1_prompts, phase1_params)

    for item, out in zip(missing_phase1, phase1_outputs):
        response      = out.outputs[0].text.strip()
        finish_reason = str(out.outputs[0].finish_reason)
        response_records[str(item["id"])] = {
            "id":              item["id"],
            "phase_used":      1,
            "response":        response,
            "answer":          extract_boxed(response),
            "finish_reason":   finish_reason,
            "uncertain":       is_uncertain(response, finish_reason),
            "n_samples":       1,
            "consensus_count": 1,
        }
    write_checkpoint(CHECKPOINT_PATH, response_records)

p1_uncertain = sum(1 for item in data_run
                   if response_records.get(str(item["id"]), {}).get("uncertain"))
print(f"Phase 1 done: {p1_uncertain}/{len(data_run)} uncertain")

# Phase 2 — single batched call for all uncertain questions
phase2_params    = make_sampling_params(
    PHASE2_MAX_TOKENS, temperature=PHASE2_TEMPERATURE,
    repetition_penalty=PHASE2_REPETITION_PENALTY,
)
phase1_uncertain = [
    item for item in data_run
    if response_records.get(str(item["id"]), {}).get("uncertain")
    and int(response_records.get(str(item["id"]), {}).get("phase_used", 0)) < 2
]

RETRY_PREFIX      = "Previous attempt was unclear. Solve this again carefully from scratch:\n\n"
MCQ_VERIFY_SUFFIX = (
    "\n\nAfter finding your answer, check each option against the problem conditions. "
    "Eliminate any letter that clearly fails. "
    "Then on the very last line write ONLY \\boxed{X}."
)

if phase1_uncertain:
    print(f"Phase 2: {len(phase1_uncertain)} uncertain x {PHASE2_N_SAMPLES} = "
          f"{len(phase1_uncertain)*PHASE2_N_SAMPLES} prompts")
    phase2_prompts = []
    for item in phase1_uncertain:
        suffix      = MCQ_VERIFY_SUFFIX if item.get("options") else ""
        prompt_text = build_chat_prompt(
            item, thinking_budget=PHASE2_THINKING_BUDGET,
            prefix=RETRY_PREFIX, suffix=suffix,
        )
        for _ in range(PHASE2_N_SAMPLES):
            phase2_prompts.append(prompt_text)

    phase2_outputs_flat = llm.generate(phase2_prompts, phase2_params)

    for q_idx, item in enumerate(phase1_uncertain):
        start          = q_idx * PHASE2_N_SAMPLES
        end            = start + PHASE2_N_SAMPLES
        samples        = [phase2_outputs_flat[j].outputs[0].text.strip()       for j in range(start, end)]
        finish_reasons = [str(phase2_outputs_flat[j].outputs[0].finish_reason) for j in range(start, end)]
        chosen         = choose_best_sample(samples, finish_reasons)
        chosen.update({"id": item["id"], "phase_used": 2})
        response_records[str(item["id"])] = chosen

    write_checkpoint(CHECKPOINT_PATH, response_records)

missing_ids = [item["id"] for item in data_run if str(item["id"]) not in response_records]
if missing_ids:
    raise RuntimeError(f"Missing responses for {len(missing_ids)} id(s): {missing_ids[:10]}")

responses = [response_records[str(item["id"])]["response"] for item in data_run]

phase_counts    = Counter(response_records[str(item["id"])].get("phase_used") for item in data_run)
uncertain_count = sum(bool(response_records[str(item["id"])].get("uncertain"))  for item in data_run)
finish_counts   = Counter(str(response_records[str(item["id"])].get("finish_reason")) for item in data_run)
length_hits     = (finish_counts.get("RequestStatus.FINISH_REASON_LENGTH", 0)
                   + finish_counts.get("length", 0))

print(f"Done: {len(responses)} responses | "
      f"phase1={phase_counts.get(1,0)} phase2={phase_counts.get(2,0)} "
      f"uncertain={uncertain_count} truncated={length_hits}")


### Generate with Transformers (for Datahub)

In [10]:
# --- Transformers generation path (disabled when using vLLM above — do not run both) ---
# prompts = []
# for item in data_run:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#         enable_thinking=True,
#     )
#     prompts.append(prompt_text)
#
# responses = []
# print(
#     f"Generating {len(prompts)} response(s), max_new_tokens={MAX_TOKENS} each "
#     "(raise MAX_TOKENS in config if answers get cut off).",
#     flush=True,
# )
#
# with torch.no_grad():
#     for prompt_text in tqdm(prompts, desc="Generate", unit="q"):
#         inputs = tokenizer(
#             prompt_text,
#             return_tensors="pt",
#             truncation=True,
#             max_length=16384,
#         ).to(_device)
#         output_ids = llm.generate(
#             **inputs,
#             max_new_tokens=MAX_TOKENS,
#             temperature=0.6,
#             top_p=0.95,
#             top_k=20,
#             min_p=0.0,
#             repetition_penalty=1.0,
#             do_sample=True,
#             pad_token_id=tokenizer.eos_token_id,
#         )
#         new_tokens = output_ids[0, inputs["input_ids"].shape[1] :]
#         responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())
#
# print(f"Done. len(responses)={len(responses)} (expected {len(data_run)}).", flush=True)
#
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data_run[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")


## 7. Score Responses

**Requires** `answer` in each row (public set). If you loaded `private.jsonl`, skip this section and only export responses.

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [11]:
print("N_QUESTIONS:", N_QUESTIONS)
print("len(data_run):", len(data_run))
print("len(responses):", len(responses) if "responses" in globals() else "missing")
print("len(results):", len(results) if "results" in globals() else "missing")
print([x["id"] for x in data_run[:10]])

N_QUESTIONS: 50
len(data_run): 50
len(responses): 50
len(results): missing
[228, 51, 563, 501, 457, 285, 209, 1116, 178, 864]


In [12]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


from judger import Judger

if "responses" not in globals():
    raise NameError(
        "`responses` is not defined. Finish the **Generate** cell above first. "
        "If you interrupted it before, that variable is never assigned — re-run Generate to completion."
    )
if len(responses) != len(data_run):
    raise ValueError(
        f"len(responses)={len(responses)} but len(data_run)={len(data_run)}. "
        "Re-run Generate for all items, or shrink data_run / N_QUESTIONS and regenerate."
    )

judger = Judger(strict_extract=False)

if not all("answer" in item for item in data_run):
    raise ValueError(
        "Scoring needs an `answer` field per row. Use `public.jsonl` for section 7, "
        "or skip scoring for the private set."
    )

print(
    f"Scoring {len(data_run)} responses — MCQ is instant; free-form `auto_judge` can take minutes "
    "on the first few items. You should see the line below update each question.",
    flush=True,
)

results = []
with tqdm(total=len(data_run), desc="Scoring", unit="q", dynamic_ncols=True) as pbar:
    for item, response in zip(data_run, responses):
        pbar.set_postfix_str(f"id={item.get('id')}", refresh=True)

        is_mcq = bool(item.get("options"))
        gold = item["answer"]

        if is_mcq:
            correct = score_mcq(response, str(gold))
        else:
            gold_list = gold if isinstance(gold, list) else [gold]
            try:
                correct = judger.auto_judge(
                    pred=response,
                    gold=gold_list,
                    options=[[]] * len(gold_list),
                )
            except Exception:
                correct = False

        results.append(
            {
                "id": item.get("id"),
                "is_mcq": is_mcq,
                "gold": gold,
                "response": response,
                "correct": correct,
            }
        )
        pbar.update(1)

print(f"Scoring complete. {len(results)} results.")

Scoring 50 responses — MCQ is instant; free-form `auto_judge` can take minutes on the first few items. You should see the line below update each question.


Scoring:   0%|          | 0/50 [00:00<?, ?q/s]

Scoring complete. 50 results.


## 8. Summary

Print accuracy broken down by question type.

In [ ]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]


def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0


print(f"MCQ       : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"Free-form : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"Overall   : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
SAVE_EVAL = DATA_MODE == "public"  # Private has no labels; set False automatically.

if "responses" not in globals():
    raise RuntimeError("Run Section 6 generation before saving results.")
if len(responses) != len(data_run):
    raise RuntimeError(f"responses length {len(responses)} != data_run length {len(data_run)}")

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

if SAVE_EVAL:
    if "results" not in globals():
        raise RuntimeError("SAVE_EVAL=True requires running Section 7 scoring first.")
    records = []
    for r in results:
        meta = response_records.get(str(r["id"]), {}) if "response_records" in globals() else {}
        records.append({
            "id": r["id"],
            "is_mcq": r["is_mcq"],
            "gold": r["gold"],
            "response": r["response"],
            "correct": r["correct"],
            "phase_used": meta.get("phase_used"),
            "uncertain": meta.get("uncertain"),
            "finish_reason": meta.get("finish_reason"),
            "consensus_count": meta.get("consensus_count"),
            "n_samples": meta.get("n_samples"),
        })
else:
    records = []
    for item, response in zip(data_run, responses):
        meta = response_records.get(str(item["id"]), {}) if "response_records" in globals() else {}
        records.append({
            "id": item["id"],
            "is_mcq": bool(item.get("options")),
            "response": response,
            "phase_used": meta.get("phase_used"),
            "uncertain": meta.get("uncertain"),
            "finish_reason": meta.get("finish_reason"),
            "consensus_count": meta.get("consensus_count"),
            "n_samples": meta.get("n_samples"),
        })

with open(out_path, "w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Saved {len(records)} records to {out_path}")


## 10. Generate Submission CSV

Convert the saved JSONL into a competition-format CSV (`id,response`).
Works for both public (evaluation) runs and private (submission) runs.
The `response` column holds the **full model trace** (competition requirement).

In [ ]:
import csv
from datetime import date

SUBMISSION_INPUT  = OUTPUT_PATH
SUBMISSION_OUTPUT = (
    REPO_ROOT / "artifacts" / "submissions"
    / f"submission_{date.today().isoformat()}.csv"
)
SUBMISSION_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

rows = []
with open(SUBMISSION_INPUT, encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        rows.append({"id": rec["id"], "response": rec["response"]})

rows.sort(key=lambda r: r["id"])

with open(SUBMISSION_OUTPUT, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "response"], quoting=csv.QUOTE_ALL)
    writer.writeheader()
    writer.writerows(rows)

print(f"Wrote {len(rows)} rows -> {SUBMISSION_OUTPUT}")


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!